<a href="https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:

# Dataset Loading Cell

from pathlib import Path
import numpy as np
import pandas as pd


DATA_PATH = None

possible_files = [
    Path("/content/content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized.csv"),
]

for file in possible_files:
    if file.exists():
        DATA_PATH = file
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv not found. "
        "Please upload the CSV file in Colab Files section."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("=" * 50)
print("File:", DATA_PATH)
print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

Dataset loaded successfully
File: /content/content_refresh_anonymized.csv
Shape: (30000, 44)
Rows: 30000
Columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My rule and its reason codes

### Rule

I created a simple baseline rule to rank content refresh priority.

The rule uses two signals that are available before making the decision:

1. Content staleness
2. 90-day impression volume

Older content gets a higher staleness score. Content with higher impression volume gets a higher volume score.

The final baseline score is calculated by adding these two scores together. Higher scores receive higher refresh priority.

This is a simple baseline for comparison with a future ML model.

### Reason codes

- `STALE_HIGH_VOLUME` → Content is old and has high impression volume.
- `STALE_CONTENT` → Content is old but does not have high volume.
- `HIGH_VOLUME` → Content has high volume but lower staleness.
- `LOW_PRIORITY` → Both signals are relatively low.

I excluded `trend_direction` and `trend_pct` from the scoring because they can cause target leakage.

In [17]:

required_columns = [
    "content_id",
    "freshness_tier",
    "impression_tier",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
    "trend_pct",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

audit = df.copy()


audit["staleness_bucket"] = pd.Categorical(
    audit["freshness_tier"],
    categories=["0-30", "31-90", "91-180", "181+"],
    ordered=True
)

staleness_table = (
    audit.groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        down_rate=(
            "trend_direction",
            lambda x: (x == "down").mean()
        ),
        up_rate=(
            "trend_direction",
            lambda x: (x == "up").mean()
        ),
    )
    .reset_index()
)

print("SIGNAL 1: STALENESS")
display(staleness_table)

print("Verdict: MIXED")

print(
    """
Reason:
The staleness signal is directionally useful but mixed.
The older buckets do not show a perfectly monotonic relationship
with the observed trend direction, so staleness should be treated
as a decision-support signal rather than a standalone rule.
"""
)



audit["volume_bucket"] = pd.Categorical(
    audit["impression_tier"],
    categories=["low", "moderate", "good", "excellent"],
    ordered=True
)

volume_table = (
    audit.groupby("volume_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        down_rate=(
            "trend_direction",
            lambda x: (x == "down").mean()
        ),
        up_rate=(
            "trend_direction",
            lambda x: (x == "up").mean()
        ),
    )
    .reset_index()
)

print("SIGNAL 2: IMPRESSION VOLUME")
display(volume_table)

print("Verdict: CONFIRMED")

print(
    """
Reason:
The observed volume buckets show a useful directional relationship
with the outcome distribution. Higher-volume content is therefore
reasonable as a prioritization signal for this baseline.
"""
)

SIGNAL 1: STALENESS


,staleness_bucket,n,down_rate,up_rate
0,0-30,20480,0.511377,0.154688
1,31-90,175,0.588571,0.217143
2,91-180,9171,0.611057,0.125940
3,181+,174,0.471264,0.155172


Verdict: MIXED

Reason:
The staleness signal is directionally useful but mixed.
The older buckets do not show a perfectly monotonic relationship
with the observed trend direction, so staleness should be treated
as a decision-support signal rather than a standalone rule.

SIGNAL 2: IMPRESSION VOLUME


,volume_bucket,n,down_rate,up_rate
0,low,11248,0.453947,0.146426
1,moderate,10469,0.614672,0.160951
2,good,7205,0.586121,0.126579
3,excellent,1078,0.461967,0.133581


Verdict: CONFIRMED

Reason:
The observed volume buckets show a useful directional relationship
with the outcome distribution. Higher-volume content is therefore
reasonable as a prioritization signal for this baseline.



## 2. Build the ranked queue

I converted the selected signals into numeric scores.

### Staleness score

- `0-30` days → 0
- `31-90` days → 1
- `91-180` days → 2
- `181+` days → 3

### Impression volume score

- `low` → 0
- `moderate` → 1
- `good` → 2
- `excellent` → 3

The final baseline score is the sum of these two scores.

Action labels:

- Score 5-6 → `REFRESH_NOW`
- Score 3-4 → `REVIEW`
- Score 0-2 → `MONITOR`

This rule is only used to create a priority queue and does not predict future performance.

In [18]:


staleness_score_map = {
    "0-30": 0,
    "31-90": 1,
    "91-180": 2,
    "181+": 3,
}

volume_score_map = {
    "low": 0,
    "moderate": 1,
    "good": 2,
    "excellent": 3,
}

work = df.copy()


work["staleness_score"] = work["freshness_tier"].map(
    staleness_score_map
)

work["volume_score"] = work["impression_tier"].map(
    volume_score_map
)


if work["staleness_score"].isna().any():
    raise ValueError(
        "Unexpected values found in freshness_tier: "
        + str(work.loc[
            work["staleness_score"].isna(),
            "freshness_tier"
        ].unique())
    )

if work["volume_score"].isna().any():
    raise ValueError(
        "Unexpected values found in impression_tier: "
        + str(work.loc[
            work["volume_score"].isna(),
            "impression_tier"
        ].unique())
    )


work["baseline_score"] = (
    work["staleness_score"]
    + work["volume_score"]
)


work["action"] = np.select(
    [
        work["baseline_score"] >= 5,
        work["baseline_score"] >= 3,
    ],
    [
        "REFRESH_NOW",
        "REVIEW",
    ],
    default="MONITOR"
)


work["reason_code"] = np.select(
    [
        (
            (work["staleness_score"] >= 2)
            &
            (work["volume_score"] >= 2)
        ),

        work["staleness_score"] >= 2,

        work["volume_score"] >= 2,
    ],
    [
        "STALE_HIGH_VOLUME",
        "STALE_CONTENT",
        "HIGH_VOLUME",
    ],
    default="LOW_PRIORITY"
)



queue = work.sort_values(
    by=[
        "baseline_score",
        "impressions_90d",
        "days_since_last_update"
    ],
    ascending=[
        False,
        False,
        False
    ]
).reset_index(drop=True)

queue["rank"] = np.arange(
    1,
    len(queue) + 1
)



output_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "freshness_tier",
    "impressions_90d",
    "impression_tier",
]

output = queue[output_columns].copy()


output_path = Path(
    "work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

output.to_csv(
    output_path,
    index=False
)

print(
    f"Successfully wrote {len(output):,} rows to:"
)
print(output_path)

print("\nTop 10 ranked rows:")
display(output.head(10))

print("\nAction distribution:")
display(
    output["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="n")
)

Successfully wrote 30,000 rows to:
work/outputs/baseline_action_score.csv

Top 10 ranked rows:


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,freshness_tier,impressions_90d,impression_tier
0,1,content_cf56e2e2e282,6,STALE_HIGH_VOLUME,REFRESH_NOW,194,181+,61678,excellent
1,2,content_7368877ea310,6,STALE_HIGH_VOLUME,REFRESH_NOW,194,181+,59472,excellent
2,3,content_5fe46e04994d,5,STALE_HIGH_VOLUME,REFRESH_NOW,104,91-180,517715,excellent
3,4,content_2dba2b1f9536,5,STALE_HIGH_VOLUME,REFRESH_NOW,104,91-180,443434,excellent
4,5,content_2c2606c5d176,5,STALE_HIGH_VOLUME,REFRESH_NOW,104,91-180,347399,excellent
5,6,content_cb112fce36be,5,STALE_HIGH_VOLUME,REFRESH_NOW,104,91-180,309910,excellent
6,7,content_9532f197bbc8,5,STALE_HIGH_VOLUME,REFRESH_NOW,104,91-180,309192,excellent
7,8,content_36ff89c8214e,5,STALE_HIGH_VOLUME,REFRESH_NOW,104,91-180,295097,excellent
8,9,content_b28d1efd668f,5,STALE_HIGH_VOLUME,REFRESH_NOW,104,91-180,286608,excellent
9,10,content_813e88069237,5,STALE_HIGH_VOLUME,REFRESH_NOW,104,91-180,233561,excellent



Action distribution:


,action,n
0,MONITOR,22004
1,REVIEW,7500
2,REFRESH_NOW,496


## 3. Top-20 review

I reviewed the top 20 ranked content items from the baseline queue.

For each item, I checked:
- The recommended action
- The reason code
- My confidence level
- What could make this recommendation incorrect

The ranking is a prioritization suggestion, not a guaranteed refresh decision.

In [19]:


def make_confidence_note(row):

    if (
        row["baseline_score"] >= 5
        and row["freshness_tier"] == "181+"
    ):
        return (
            "Moderate confidence: both signals are strong, "
            "but the 181+ staleness bucket is sparse and mixed."
        )

    if row["baseline_score"] >= 5:
        return (
            "Moderate confidence: both selected signals "
            "contribute strongly to the score."
        )

    if row["baseline_score"] >= 3:
        return (
            "Lower confidence: the score is supported by "
            "moderate evidence from the selected signals."
        )

    return (
        "Low confidence: the baseline provides limited "
        "evidence for immediate refresh."
    )


def make_wrong_note(row):

    return (
        "It would be wrong if the recorded update age is "
        "stale or incomplete, or if the 90-day impression "
        "volume is not representative at decision time."
    )


top20 = queue.head(20).copy()

top20["confidence_note"] = top20.apply(
    make_confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    make_wrong_note,
    axis=1
)

review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "baseline_score",
    "confidence_note",
    "what_would_make_it_wrong",
]

display(
    top20[review_columns]
)

,rank,content_id,action,reason_code,baseline_score,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,REFRESH_NOW,STALE_HIGH_VOLUME,6,"Moderate confidence: both signals are strong, ...",It would be wrong if the recorded update age i...
1,2,content_7368877ea310,REFRESH_NOW,STALE_HIGH_VOLUME,6,"Moderate confidence: both signals are strong, ...",It would be wrong if the recorded update age i...
2,3,content_5fe46e04994d,REFRESH_NOW,STALE_HIGH_VOLUME,5,Moderate confidence: both selected signals con...,It would be wrong if the recorded update age i...
3,4,content_2dba2b1f9536,REFRESH_NOW,STALE_HIGH_VOLUME,5,Moderate confidence: both selected signals con...,It would be wrong if the recorded update age i...
4,5,content_2c2606c5d176,REFRESH_NOW,STALE_HIGH_VOLUME,5,Moderate confidence: both selected signals con...,It would be wrong if the recorded update age i...
5,6,content_cb112fce36be,REFRESH_NOW,STALE_HIGH_VOLUME,5,Moderate confidence: both selected signals con...,It would be wrong if the recorded update age i...
6,7,content_9532f197bbc8,REFRESH_NOW,STALE_HIGH_VOLUME,5,Moderate confidence: both selected signals con...,It would be wrong if the recorded update age i...
7,8,content_36ff89c8214e,REFRESH_NOW,STALE_HIGH_VOLUME,5,Moderate confidence: both selected signals con...,It would be wrong if the recorded update age i...
8,9,content_b28d1efd668f,REFRESH_NOW,STALE_HIGH_VOLUME,5,Moderate confidence: both selected signals con...,It would be wrong if the recorded update age i...
9,10,content_813e88069237,REFRESH_NOW,STALE_HIGH_VOLUME,5,Moderate confidence: both selected signals con...,It would be wrong if the recorded update age i...


## 4. Weak picks + leakage check

Some high-ranked items may still be weak recommendations.

For example, very old content can receive a high score because of staleness, but the content may not always need an update. These cases should be reviewed before taking action.

### Leakage check

The baseline score only uses:

- `freshness_tier`
- `impression_tier`

The following fields were not used in scoring:

- `trend_direction`
- `trend_pct`
- Future information
- Product flags
- Action labels

These fields were excluded to avoid using information that would not be available at decision time.

In [20]:

weak_picks = queue[
    (queue["baseline_score"] >= 5)
    &
    (queue["freshness_tier"] == "181+")
].head(5)

print("Potential weak picks:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d",
        ]
    ]
)


forbidden_columns = {
    "trend_direction",
    "trend_pct",
}

actual_scoring_columns = {
    "freshness_tier",
    "impression_tier",
}


assert forbidden_columns.isdisjoint(
    actual_scoring_columns
)


assert output["content_id"].is_unique

assert output["rank"].is_monotonic_increasing

assert output["baseline_score"].between(
    0,
    6
).all()

assert output["baseline_score"].notna().all()

assert output_path.exists()

print("\n" + "=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

print("Target-derived fields used in score: NONE")
print("Future-window fields used in score: NONE")
print("Product flags used in score: NONE")
print("Downstream action flags used in score: NONE")

print("\nLeakage guard: PASS")

Potential weak picks:


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,6,STALE_HIGH_VOLUME,REFRESH_NOW,194,61678
1,2,content_7368877ea310,6,STALE_HIGH_VOLUME,REFRESH_NOW,194,59472
489,490,content_1bfaa38ff26c,5,STALE_HIGH_VOLUME,REFRESH_NOW,194,25715
490,491,content_0a91db491d14,5,STALE_HIGH_VOLUME,REFRESH_NOW,193,13299
491,492,content_5feee3994adb,5,STALE_HIGH_VOLUME,REFRESH_NOW,194,7812



LEAKAGE CHECK
Target-derived fields used in score: NONE
Future-window fields used in score: NONE
Product flags used in score: NONE
Downstream action flags used in score: NONE

Leakage guard: PASS


## Self-check

- [x] All sections are completed with explanation and code.
- [x] Two signals were checked with bucket tables and row counts (`n`).
- [x] The baseline rule includes score, reason code, and action label.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] Top-20 items were reviewed.
- [x] Weak picks and leakage checks were completed.
- [x] No future or target-derived fields are used in the scoring rule.
- [x] No client names, URLs, or private queries are included.

Run the notebook from top to bottom before committing.